# 07 · Power BI Export: star schema

Writes the small, tidy tables the dashboard needs (notebook 08 builds the report itself) to `powerbi/data/`.
Every number in the dashboard comes from these files, and every file comes from the warehouse built in notebooks 02–06.

| Table | Grain | Used for |
|---|---|---|
| `dim_trip` | one row per trip | season / route slicers, trip attributes |
| `fact_trip` | one row per trip | energy, consumption, climate load, regeneration |
| `fact_sag_event` | one voltage-sag event | resistance vs temperature, flagged events |
| `fact_warmup` | minute since start (winter trips) | heating warm-up profile |
| `fact_regen_limit` | battery-temperature band | regeneration limit of a cold battery |
| `fact_soc_test` | 5-second step of a test trip | SoC: recorded vs model estimate |
| `fact_model` | task × model × split | benchmark of 8 models + stacked ensemble |
| `fact_driver` | task × feature (top 6) | what drives each model |
| `fact_anomaly` | one flag | anomaly register with root-cause notes |

In [1]:
from pathlib import Path

import duckdb
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DB_PATH = ROOT / "data" / "warehouse" / "volvo.duckdb"
PBI_DATA = ROOT / "powerbi" / "data"
PBI_DATA.mkdir(parents=True, exist_ok=True)
con = duckdb.connect(str(DB_PATH), read_only=True)
tables = {}

In [2]:
tables["dim_trip"] = con.sql("""
    SELECT t.trip_id,
           replace(t.trip_id, 'Trip', '')                    AS trip_label,
           upper(substr(t.season, 1, 1)) || substr(t.season, 2) AS season,
           t.route, t.weather,
           strftime(t.start_time, '%Y-%m-%d')                AS trip_date,
           round(e.distance_km_integrated, 2)                AS distance_km,
           round(t.duration_min, 1)                          AS duration_min,
           round(t.ambient_temp_start_c, 1)                  AS ambient_start_c,
           t.batt_temp_start_c,
           (t.trip_id IN (SELECT trip_id FROM model.test_trips))::INT AS is_test_trip
    FROM clean.trips t JOIN signal.trip_energy e USING (trip_id)
    ORDER BY t.trip_id
""").df()

tables["fact_trip"] = con.sql("""
    WITH d AS (SELECT c.trip_id, avg(c.speed_kmh) AS mean_speed_kmh, avg(c.ambient_temp_c) AS ambient_mean_c
               FROM clean.trip_signals c JOIN signal.samples s USING (trip_id, row_idx)
               WHERE s.phase = 'driving' GROUP BY 1)
    SELECT e.trip_id,
           round(e.distance_km_integrated, 3) AS distance_km,
           round(e.energy_out_kwh, 4)          AS energy_out_kwh,
           round(e.energy_regen_kwh, 4)        AS energy_regen_kwh,
           round(e.energy_net_drive_kwh, 4)    AS energy_net_kwh,
           round(e.heating_kwh, 4)             AS heating_kwh,
           round(e.aircon_kwh, 4)              AS aircon_kwh,
           round(e.energy_charged_kwh, 4)      AS energy_charged_kwh,
           round(d.mean_speed_kmh, 1)          AS mean_speed_kmh,
           round(d.ambient_mean_c, 1)          AS ambient_mean_c,
           round(r.rise_c, 1)                  AS batt_temp_rise_c,
           round(a.expected_kwh_per_100km, 2)  AS expected_kwh_per_100km,
           round(a.isolation_score, 3)         AS isolation_score
    FROM signal.trip_energy e JOIN d USING (trip_id) JOIN signal.temp_rise r USING (trip_id)
    JOIN anomaly.trip_scores a USING (trip_id)
    ORDER BY e.trip_id
""").df()

tables["fact_sag_event"] = con.sql("""
    SELECT row_number() OVER (ORDER BY trip_id, start_s) AS event_id, trip_id,
           round(start_s, 1) AS start_s, batt_temp_c, round(soc_pct, 1) AS soc_pct,
           round(-current_at_max_sag_a, 1) AS discharge_current_a, round(max_sag_v, 2) AS sag_v,
           round(apparent_resistance_mohm, 1) AS resistance_mohm, round(expected_resistance_mohm, 1) AS expected_resistance_mohm,
           CASE WHEN flag THEN 'Flagged' ELSE 'Normal' END AS status
    FROM anomaly.resistance_events ORDER BY event_id
""").df()

tables["fact_warmup"] = con.sql("""
    SELECT floor(time_s / 60)::INT AS minute, round(avg(heating_power_can_kw), 3) AS heating_kw,
           round(avg(cabin_temp_c), 2) AS cabin_temp_c, count(DISTINCT trip_id) AS trips
    FROM clean.trip_signals WHERE series = 'B' AND time_s < 1800
    GROUP BY 1 ORDER BY 1
""").df()

tables["fact_regen_limit"] = con.sql("""
    SELECT (floor(c.batt_temp_c / 5) * 5)::INT AS band_low_c,
           printf('%d–%d', (floor(c.batt_temp_c / 5) * 5)::INT, (floor(c.batt_temp_c / 5) * 5 + 5)::INT) AS batt_temp_band,
           round(quantile_cont(c.batt_current_a, 0.99), 1) AS regen_current_p99_a, count(*) AS samples
    FROM clean.trip_signals c JOIN signal.samples s USING (trip_id, row_idx)
    WHERE s.phase = 'driving' AND c.batt_current_a > 0
    GROUP BY 1, 2 HAVING count(*) >= 500 ORDER BY 1
""").df()

tables["fact_soc_test"] = con.sql("""
    SELECT trip_id, round(floor(time_s / 5) * 5 / 60, 3) AS minute,
           round(avg(soc_pct), 2) AS soc_recorded, round(avg(soc_pred), 2) AS soc_estimated
    FROM model.soc_test_predictions GROUP BY 1, 2 ORDER BY 1, 2
""").df()

In [3]:
m = con.sql("SELECT * FROM model.metrics").df()
m["is_selected"] = m.is_selected.astype(int)
m["split"] = m.split.replace({"cross-validation": "cross-val"})
tables["fact_model"] = m.round(3)[["task", "split", "stage", "model", "family", "mae", "rmse", "r2", "is_selected"]]

PRETTY = {"voltage_mean_60s_v": "Voltage, 60 s", "voltage_smooth_v": "Voltage", "v_relaxed": "Relaxed voltage",
          "batt_temp_c": "Battery temp", "current_smooth_a": "Current", "ambient_temp_c": "Outside temp",
          "current_mean_60s_a": "Current, 60 s", "power_smooth_kw": "Power", "heating_kw": "Heating",
          "climb_m_per_km": "Climb", "aircon_kw": "A/C", "mean_abs_accel_ms2": "Acceleration",
          "mean_speed_kmh": "Speed", "speed_std_kmh": "Speed spread"}
d = con.sql("SELECT * FROM model.feature_importance").df()
d["feature"] = d.feature.map(PRETTY)
d["share"] = d.importance / d.groupby("task").importance.transform("sum")
d["rank"] = d.groupby("task").importance.rank(ascending=False, method="first").astype(int)
tables["fact_driver"] = d[d["rank"] <= 6].round(4)[["task", "feature", "importance", "share", "rank"]]  # top 6 inputs per model

a = con.sql("SELECT flag_id, detector, level, trip_id, score, reason FROM anomaly.flags").df()
a["detector_short"] = a.detector.str.split(" · ").str[1].str.capitalize()
tables["fact_anomaly"] = a

In [4]:
for name, frame in tables.items():
    frame.to_csv(PBI_DATA / f"{name}.csv", index=False, encoding="utf-8")
assert set(tables["fact_trip"].trip_id) == set(tables["dim_trip"].trip_id)
for name in ["fact_sag_event", "fact_soc_test", "fact_anomaly"]:
    assert set(tables[name].trip_id) <= set(tables["dim_trip"].trip_id), name
con.close()
pd.DataFrame([{"table": k, "rows": len(v), "columns": v.shape[1]} for k, v in tables.items()])

,table,rows,columns
0,dim_trip,70,11
1,fact_trip,70,13
2,fact_sag_event,556,10
3,fact_warmup,30,4
4,fact_regen_limit,7,4
5,fact_soc_test,3620,4
6,fact_model,42,9
7,fact_driver,12,5
8,fact_anomaly,40,7


**Integrity:** every fact table's `trip_id` exists in `dim_trip`, and `fact_trip` has exactly one row per trip.
The values match the notebooks they come from, because each table is a direct query of the warehouse.